# Import libraries

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

ModuleNotFoundError: No module named 'matplotlib'

# Load AI-filtered Bluesky posts

In [ ]:
DATA_PATH = Path("../data/v2/bsky_AI_filtered_posts.csv")
PLOTS_DIR = Path("../data/v2/plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"], utc=True, errors="coerce", format="mixed")

analysis_df = (
    df.dropna(subset=["Date", "emotion_label"])
    .sort_values("Date")
    .copy()
)

print(f"Loaded {len(df):,} AI-filtered posts.")
print(f"Using {len(analysis_df):,} posts with valid dates and emotion labels.")
print(f"Date range: {analysis_df['Date'].min().date()} to {analysis_df['Date'].max().date()}")
display(analysis_df["emotion_label"].value_counts().rename_axis("emotion").reset_index(name="rows"))

# Plot emotions distributions accross time

In [ ]:
# Use 45-day aggregation because there may not be enough data for single-day analysis.
emotion_counts_45d = (
    analysis_df
    .groupby([pd.Grouper(key="Date", freq="45D"), "emotion_label"])
    .size()
    .rename("post_count")
    .reset_index()
)

emotion_counts_wide_45d = (
    emotion_counts_45d
    .pivot(index="Date", columns="emotion_label", values="post_count")
    .fillna(0)
    .sort_index()
)

display(emotion_counts_wide_45d.head())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

for emotion in emotion_counts_wide_45d.columns:
    ax.plot(
        emotion_counts_wide_45d.index,
        emotion_counts_wide_45d[emotion],
        marker="o",
        linewidth=2,
        label=emotion,
    )

ax.set_title("AI-Related Bluesky Posts by Emotion, 45-Day Counts")
ax.set_xlabel("Date")
ax.set_ylabel("Number of posts")
ax.grid(True, alpha=0.3)
ax.legend(title="Emotion", ncol=2)

fig.autofmt_xdate()
plt.tight_layout()

plot_path = PLOTS_DIR / "bsky_ai_emotion_counts_45d.png"
fig.savefig(plot_path, dpi=300, bbox_inches="tight")
print(f"Saved plot to {plot_path}")
plt.show()